In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import folium
import ckanapi as ck
import os
from dotenv import load_dotenv

#### API Key Loading
If you don't have an API key, get one here: https://api.data.gov/signup/

This block is just to load in your api key, in order to use it make a file titled '.env' like literally just that, no actual file name just '.env' and put this in there

API_KEY = [insert your api key here]

In [4]:
load_dotenv()
API_KEY = os.getenv('API_KEY')

#### Wrapper Function for Searching CKAN API
I wrote this function for searching the CKAN API since I think it's really annoying to do by hand. Honestly probably don't even need this function but I'm so fed up with the CKAN API that I have this now.

Note about the output of this function: It outputs a giant dictionary that's just a lot of JSON. To actually get the data instead of just more JSON metadata, look any dictionary that has a metadata tag about a **datastore being True** and then find the **package id** and thats how you get the actual data instead of more useless JSON metadata

In [119]:
#Initial CKAN request (I guess a lot of governments use CKAN for their open access data

def search_package(url,query):
    '''
    Queries the CKAN API and returns the results as a JSON Dictionary.

    Args:
        url: CKAN API URL
        query: Query string

    Returns:
        results: JSON Dictionary
    '''

    #Sidenote: I wrote the docstring myself using guidelines from https://www.geeksforgeeks.org/python/python-docstrings/
    #none of my code was written by AI lol
    request = ck.RemoteCKAN(url, apikey=API_KEY)
    result = request.call_action('package_search', {'q':query})
    return result

#### Finding the Actual Data
This entire code block is literally just looking for the actual data instead of the random JSON metadata. The final output of this cell, 'data_dicts' is literally just to display the name of the sub-dataset and the resource ID for it. You need the resource ID to call the actual data, NOT the package ID. The package ID will just spit out a bunch of random metadata no one cares about.

**Also really important note:** I'm actually using a different dataset here: https://data.chhs.ca.gov/dataset/hospital-inpatient-characteristics-by-patient-county-of-residence

The one we talked about using before was the dataset *by facility* meaning by hospital. We don't really need the hospital stuff since we don't really care about the hospital these cases were from, just the county. This dataset also splits all the data into separate files for sex, race, type of care, ect. which is why I made a dictionary to link 9 resource IDs as opposed to just one.

In [189]:
ca_hhs_url = 'https://data.chhs.ca.gov/'
inpatient_query = 'hospital inpatient characteristics - MDC by patient county'
#See above Markdown text to see how I got this ugly subsetting
#If you want to see what I had to look through to get the subsetting, just run the function without any of the bracket stuff
hospital_patient_json = search_package(ca_hhs_url, inpatient_query)['results'][0]['resources'][1:9]

#Making individual lists for the name of the Resource and the Resource ID
data_names = [dct['name'] for dct in hospital_patient_json]
data_ids = [dct['resource_id'] for dct in hospital_patient_json]

#Putting them into one dictionary and calling it
data_dicts = {data_names[i]:data_ids[i] for i in range(len(data_names))}
data_dicts

[{'accrual_periodicity': 'annually',
  'additional_information': '',
  'additional_limitations': '',
  'author': 'Department of Health Care Access and Information, Healthcare Analytics Branch',
  'author_email': None,
  'citation': '',
  'contact_email': 'dataandreports@hcai.ca.gov',
  'creator_user_id': 'fd826e35-dce7-4e6a-bf68-212389b73fc2',
  'data_collection_tool': 'Patient Data Reporting',
  'data_license': 'Terms of Use',
  'de_identification_method': 'https://chhsdata.github.io/dataplaybook/resource_library/#datade-id',
  'geo_coverage': 'Statewide',
  'geographic_granularity': 'location point',
  'id': 'ed5d21ce-9ec4-44ca-a482-2f4c767e0528',
  'isopen': False,
  'language': 'English',
  'license_id': None,
  'license_title': None,
  'limitations': 'Limitations',
  'maintainer': None,
  'maintainer_email': None,
  'metadata_created': '2018-03-14T17:40:29.530515',
  'metadata_modified': '2025-11-07T01:38:03.862887',
  'name': 'hospital-inpatient-characteristics-by-facility-pivot-

#### Actually Getting the Data Now
This tiny code block actually pulls the data from the API as opposed to just looking for the information to locate the data. Basically the previous code block was just getting the resource IDs for the stuff we want and this one actually gets the stuff we want. As for the "stuff we want" I just kinda grabbed a few that seem useful, but can add more if needed. Is this making a lot of API calls? Yeah kinda. Do we need to make them more than once ever? No so it should be fine I hope lol.

In [122]:
data_request = ck.RemoteCKAN('https://data.chhs.ca.gov/', apikey=API_KEY)
MDC_data_dict = data_request.call_action('datastore_search', {'resource_id':'57eaaf0f-6ba9-428a-9498-07070c1e55cc',
                                              'limit':32000})
sex_data_dict = data_request.call_action('datastore_search', {'resource_id':'a9eaf75d-f45f-48f1-80d6-493eef9e7968',
                                              'limit':32000})
race_data_dict = data_request.call_action('datastore_search', {'resource_id':'83a8cd12-434a-4a8d-9721-ca75899568bd',
                                              'limit':32000})
care_data_dict = data_request.call_action('datastore_search', {'resource_id':'58bed582-14ae-4521-b43f-e7682edb0819',
                                              'limit':32000})

#### Pandas Data Cleanup
Just a bunch of pandas data cleaning, not really important to look at tbh but if you want it explained later just lmk

In [190]:
def inpatient_data_cleanup(dct, year, drops = None, renames=None):
    '''
    Takes a dict of JSON data and converts it into a pandas dataframe, dropping list specified columns and renaming columns specified by rename_key_dict.

    Parameters:
        dct: Dict of JSON data
        drops: Optional list of column names to drop
        renames: Optional dict of column names to rename
        year

    Returns:
        Pandas dataframe
    '''

    df = pd.DataFrame(dct['records'])

    if drops is not None and type(drops) is list:
        df = df.drop(columns= drops)
    elif drops is not None and type(drops) is not list:
        raise TypeError('drops must be a list')

    if type(renames) is dict and renames is not None:
        df = df.rename(columns=renames)
    elif type(renames) != dict and renames is not None:
        raise TypeError('renames must be a dictionary or use string "inpatient_default" for inpatient data default!')


    if (year >= 2012 or year <= 2024) and type(year) is int:
        if (renames == 'default' or type(renames) is dict):
            df['Year'] = df['Year'].apply(pd.to_numeric)
            df = df[df['Year'] == year]
        else:
            df['dsch_yr'] = df['dsch_yr'].apply(pd.to_numeric)
            df = df[df['dsch_yr'] == year]
    elif type(year) is not int:
        raise TypeError('year must be an integer!')
    elif year < 2012 or year > 2024:
        raise ValueError('year must be between 2012 and 2024!')

    return df

In [181]:
MDC_data = inpatient_data_cleanup(MDC_data_dict, 2024, ['AnnotationCode','AnnotationDesc','_id','mdc'], {'mdc_desc':'MDC'})
MDC_data[(MDC_data['MDC'] == 'NERVOUS SYSTEM, DISEASES & DISORDERS') |
         (MDC_data['MDC'] == 'RESPIRATORY SYSTEM, DISEASES & DISORDERS') |
         (MDC_data['MDC'] == 'CIRCULATORY SYSTEM, DISEASES & DISORDERS') |
         (MDC_data['MDC'] == 'DIGESTIVE SYSTEM, DISEASES & DISORDERS')]

,County,Year,MDC,Discharges
312,Alameda,2024,"NERVOUS SYSTEM, DISEASES & DISORDERS",9125
315,Alameda,2024,"RESPIRATORY SYSTEM, DISEASES & DISORDERS",9254
316,Alameda,2024,"CIRCULATORY SYSTEM, DISEASES & DISORDERS",13941
317,Alameda,2024,"DIGESTIVE SYSTEM, DISEASES & DISORDERS",8593
474,Alpine,2024,"NERVOUS SYSTEM, DISEASES & DISORDERS",4
...,...,...,...,...
18928,Yolo,2024,"DIGESTIVE SYSTEM, DISEASES & DISORDERS",1247
19252,Yuba,2024,"NERVOUS SYSTEM, DISEASES & DISORDERS",629
19255,Yuba,2024,"RESPIRATORY SYSTEM, DISEASES & DISORDERS",660
19256,Yuba,2024,"CIRCULATORY SYSTEM, DISEASES & DISORDERS",1143


In [182]:
sex_data = inpatient_data_cleanup(sex_data_dict, 2024, ['AnnotationCode','AnnotationDesc','_id'], renames = 'inpatient_default')
sex_data

,County,Year,Sex,Discharges
36,Alameda,2024,Female,71064
37,Alameda,2024,Male,56894
38,Alameda,2024,"Unknown, Invalid",27
63,Alpine,2024,Female,None
64,Alpine,2024,Male,None
...,...,...,...,...
1981,Yolo,2024,Male,7633
1982,Yolo,2024,"Unknown, Invalid",4
2010,Yuba,2024,Female,5368
2011,Yuba,2024,Male,4073


In [183]:
race_data = inpatient_data_cleanup(race_data_dict, 2024, ['AnnotationCode','AnnotationDesc','_id'], {'race_grp1':'Race'})
race_data

,County,Year,Race,Discharges
99,Alameda,2024,American Indian/Alaska Native,269
100,Alameda,2024,Asian,28727
101,Alameda,2024,Black,22319
102,Alameda,2024,Hispanic,27218
103,Alameda,2024,Invalid/Blank,10
...,...,...,...,...
6105,Yuba,2024,Multi-racial,81
6106,Yuba,2024,Native Hawaiian/Other Pacific Islander,52
6107,Yuba,2024,Other,339
6108,Yuba,2024,Unknown,127


In [184]:
care_data = inpatient_data_cleanup(care_data_dict, 2024, ['_id'], {'typcare1':'Care'})
care_data

,County,Year,Care,Discharges
64,Alameda,2024,Acute Care,120153
65,Alameda,2024,Chemical Dependency Recovery Care,85
66,Alameda,2024,Invalid/Blank,14
67,Alameda,2024,Physical Rehabilitation Care,1375
68,Alameda,2024,Psychiatric Care,5740
...,...,...,...,...
3855,Yuba,2024,Acute Care,8873
3856,Yuba,2024,Chemical Dependency Recovery Care,2
3857,Yuba,2024,Physical Rehabilitation Care,93
3858,Yuba,2024,Psychiatric Care,424


#### Issue

Big problem now: I think all of the api stuff I just did was completely pointless. I used the other dataset (by county instead of by facility) because it has an api endpoint, unlike the facility one, except it only has *discharge* data, not data on the actual occupancy and doesn't include deaths. The facility one has all that data and even more but it doesn't have an API endpoint. In the next code block I'll clean up the excel data from the facility data but import it into python by just downloading the excel file.

Note: The facility data does actually have an api endpoint, but its all from 2021 or before, meaning we either have to choose to use very outdated data or use data heavily biased by the COVID-19 pandemic.

In [217]:
facility_data = pd.read_excel('./data/2024pddpivot.xlsx', sheet_name='Data')
facility_data = facility_data.filter(items = ['COUNTY',"Sex_Male", 'Sex_Female', 'Sex_Other_Unknown', 'Age_0_09', 'Age_10_19' 'Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59', 'Age_60_69', 'Age_70_79', 'Age_80_89', 'Age_Other_Unknown', 'eth_Hispanic', 'eth_Nonhispanic', 'eth_Other', 'racegrp_aman', 'racegrp_asian', 'racegrp_nhpi', 'racegrp_black', 'racegrp_white', 'racegrp_unknown', 'racegrp_other', 'racegrp_multirace', 'dx_Infectious', 'dx_Circulatory', 'dx_Respiratory', 'dx_Digestive', 'dx_Infectious', 'dx_Skin', 'dx_MentalHealth'])
for var in list(facility_data.columns)[1:len(facility_data.columns)]:
    facility_data[var] = facility_data[var].apply(pd.to_numeric)
    facility_data[var] = facility_data.groupby('COUNTY')[var].transform('sum')
facility_data = facility_data.drop_duplicates()
#need to fix ages
facility_data

C:\Users\Zachary\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,COUNTY,Sex_Male,Sex_Female,Sex_Other_Unknown,Age_0_09,Age_30_39,Age_40_49,Age_50_59,Age_60_69,Age_70_79,...,racegrp_white,racegrp_unknown,racegrp_other,racegrp_multirace,dx_Infectious,dx_Circulatory,dx_Respiratory,dx_Digestive,dx_Skin,dx_MentalHealth
0,ALAMEDA,63652,77486,36.0,21775.0,18809.0,10601.0,12349.0,18971.0,20076.0,...,43103.0,2443.0,37004.0,2680.0,16134.0,16297.0,7943.0,11238.0,1816.0,13707.0
18,AMADOR,1187,1372,0.0,220.0,178.0,118.0,203.0,465.0,574.0,...,2271.0,55.0,108.0,50.0,648.0,339.0,169.0,239.0,39.0,48.0
19,BUTTE,15440,17791,0.0,2735.0,3092.0,2630.0,3641.0,6283.0,6413.0,...,25757.0,469.0,4374.0,302.0,5439.0,4635.0,1933.0,3393.0,490.0,1122.0
23,CALAVERAS,430,486,0.0,0.0,42.0,61.0,99.0,211.0,232.0,...,864.0,17.0,21.0,0.0,113.0,130.0,132.0,146.0,43.0,41.0
24,COLUSA,355,358,0.0,0.0,55.0,53.0,128.0,160.0,141.0,...,627.0,3.0,0.0,41.0,176.0,98.0,86.0,60.0,38.0,27.0
25,CONTRA COSTA,35763,47551,9.0,11232.0,10461.0,5573.0,7337.0,11884.0,13802.0,...,43722.0,1835.0,15411.0,2028.0,9553.0,10926.0,4499.0,8030.0,1284.0,4920.0
33,DEL NORTE,1062,1237,0.0,263.0,200.0,168.0,237.0,423.0,432.0,...,1865.0,36.0,140.0,57.0,522.0,268.0,114.0,222.0,56.0,48.0
34,EL DORADO,3239,3712,0.0,682.0,736.0,530.0,612.0,1148.0,1358.0,...,5867.0,247.0,593.0,66.0,748.0,803.0,628.0,831.0,161.0,527.0
37,FRESNO,49784,64353,2.0,16045.0,15198.0,10287.0,12132.0,16635.0,15993.0,...,87692.0,752.0,7721.0,732.0,8822.0,14117.0,5660.0,10087.0,1797.0,5211.0
47,GLENN,162,175,0.0,0.0,12.0,13.0,27.0,66.0,105.0,...,294.0,20.0,23.0,0.0,24.0,19.0,60.0,17.0,15.0,2.0
